In [30]:
from typing import List, Callable, Iterable, Tuple
import math
import operator
import random
import tqdm

In [8]:
Tensor = list
# 1. Definimos la función sigmoide si no se puede importar
def sigmoid(x: float) -> float:
    return 1 / (1 + math.exp(-x))

# Definición de la aproximación de la inversa de la CDF normal
def inverse_normal_cdf(p: float, mu: float = 0, sigma: float = 1) -> float:
    """Si no es estándar, computa el estándar y escala"""
    if mu != 0 or sigma != 1:
        return mu + sigma * inverse_normal_cdf(p)

    low_p, low_z = 0.0, -10.0  # normal_cdf(-10) es casi 0
    hi_p, hi_z = 1.0, 10.0     # normal_cdf(10) es casi 1
    
    while hi_z - low_z > 0.0001:
        mid_z = (low_z + hi_z) / 2
        mid_p = normal_cdf(mid_z)
        if mid_p < p:
            low_z, low_p = mid_z, mid_p
        else:
            hi_z, hi_p = mid_z, mid_p

    return mid_z

def normal_cdf(x: float, mu: float = 0, sigma: float = 1) -> float:
    return (1 + math.erf((x - mu) / math.sqrt(2) / sigma)) / 2

Vector = List[float]

def dot(v: Vector, w: Vector) -> float:
    """Computa v_1 * w_1 + ... + v_n * w_n"""
    assert len(v) == len(w), "Los vectores deben tener la misma longitud"
    return sum(v_i * w_i for v_i, w_i in zip(v, w))


In [9]:
def shape(tensor: Tensor) -> List[int]:
    sizes: List[int] = []
    while isinstance(tensor, list):
        sizes.append(len(tensor))
        tensor = tensor[0]
    return sizes

assert shape([1, 2, 3]) == [3]
assert shape([[1, 2], [3, 4], [5, 6]]) == [3, 2]

In [10]:
def is_1d(tensor: Tensor) -> bool:
    """Si el tensor[0] es una lista, es un tensor de orden superior. De lo contrario, el tensor es 1-dimensional (es decir, un vector)."""
    return not isinstance(tensor[0], list)

assert is_1d([1, 2, 3])
assert not is_1d([[1, 2], [3, 4]])

In [11]:
def tensor_sum(tensor: Tensor) -> float:
    """Suma todos los valores en el tensor"""
    if is_1d(tensor):
        return sum(tensor)
    else:
        return sum(tensor_sum(tensor_i) for tensor_i in tensor)
    
assert tensor_sum([1, 2, 3]) == 6
assert tensor_sum([[1, 2], [3, 4]]) == 10

In [12]:
def tensor_apply(f: Callable[[float], float], tensor: Tensor) -> Tensor:
    """Se aplica a los elementos"""
    if is_1d(tensor):
        return [f(x) for x in tensor]
    else:
        return[tensor_apply(f, tensor_i) for tensor_i in tensor]
    
assert tensor_apply(lambda x: x + 1, [1, 2, 3]) == [2, 3, 4]
assert tensor_apply(lambda x: 2 * x, [[1, 2], [3, 4]]) == [[2, 4], [6, 8]]

In [13]:
def zeros_like(tensor: Tensor) -> Tensor:
    return tensor_apply(lambda _: 0.0, tensor)

assert zeros_like([1, 2, 3]) == [0, 0, 0]
assert zeros_like([[1, 2], [3, 4]]) == [[0, 0], [0, 0]]

In [14]:
def tensor_combine(f: Callable[[float, float], float], t1: Tensor, t2: Tensor) -> Tensor:
    """Se aplica a los elementos correspondientes de t1 y t2"""
    if is_1d(t1):
        return [f(x, y) for x, y in zip(t1, t2)]
    else:
        return [tensor_combine(f, t1_i, t2_i) for t1_i, t2_i in zip(t1, t2)]
    
assert tensor_combine(operator.add, [1, 2, 3], [4, 5, 6]) == [5, 7, 9]
assert tensor_combine(operator.mul, [1, 2, 3], [4, 5, 6]) == [4, 10, 18]

In [15]:
class Layer:
    """Nuestras redes neuronales estarán compuestas por Capas, cada una de las cuales Sabe cómo hacer algún cálculo sobre sus entradas en el "forward" Direccionar y propagar gradientes en la dirección "hacia atrás"."""

    def forward(self, input):
        """Tenga en cuenta la falta de tipos. No vamos a ser prescriptivos Acerca de qué tipos de entradas pueden tomar las capas y qué tipos De los productos pueden regresar."""
        raise NotImplementedError

    def backward(self, gradient):
        """Del mismo modo, no vamos a ser prescriptivos sobre lo que El gradiente parece. Depende de ti el usuario asegurarse Que estás haciendo las cosas con sensatez."""
        raise NotImplementedError
    
    def params(self) -> Iterable[Tensor]:
        """Devuelve los parámetros de esta capa. La implementación por defecto No devuelve nada, de modo que si tienes una capa sin parámetros No tienes que implementar esto."""
        return ()
    
    def grads(self) -> Iterable[Tensor]:
        """Devuelve los gradientes, en el mismo orden que params()."""
        return ()

In [16]:
class Sigmoid(Layer):
    def forward(self, input: Tensor) -> Tensor:
        self.sigmoids = tensor_apply(sigmoid, input)
        return self.sigmoids
    
    def backward(self, gradient: Tensor) -> Tensor:
        # La derivada de la sigmoide es: f(x) * (1 - f(x))
        return tensor_combine(lambda sig, grad: sig * (1 - sig) * grad, self.sigmoids, gradient)

In [17]:
def random_uniform(*dims: int) -> Tensor:
    if len(dims) == 1:
        return [random.random() for _ in range(dims[0])]
    else:
        return [random_uniform(*dims[1:]) for _ in range(dims[0])]
    
def random_normal(*dims: int, mean: float = 0.0, variance: float = 1.0) -> Tensor:
    if len(dims) == 1:
        return[mean + variance * inverse_normal_cdf(random.random()) for _ in range(dims[0])]
    else:
        return [random_normal(*dims[1:], mean=mean, variance=variance) for _ in range(dims[0])]
    
assert shape(random_uniform(2, 3, 4)) == [2, 3, 4]
assert shape(random_normal(5, 6, mean=10)) == [5, 6]   

In [18]:
def random_tensor(*dims: int, init: str = 'normal') -> Tensor:
    if init == 'normal':
        return random_normal(*dims)
    elif init == 'uniform':
        return random_uniform(*dims)
    elif init == 'xavier':
        # La lógica Xavier: varianza depende de la suma de dimensiones
        # Asegúrate de pasar variance= como argumento nombrado
        variance = 2.0 / sum(dims) 
        return random_normal(*dims, variance=variance) # <--- Cambiado aquí
    else:
        raise ValueError(f"unknown init: {init}")

In [19]:
class Linear(Layer):
    def __init__(self, input_dim: int, output_dim: int, init: str = 'xavier') -> None:
        """Una capa de neuronas output_dim, cada una con pesos input_dim (y un sesgo)."""
        self.input_dim = input_dim
        self.output_dim = output_dim
        # self.w[o] es los pesos para la neurona o
        self.w = random_tensor(output_dim, input_dim, init=init)
        # self.b[o] es del termino de sesgo para la neuron o
        self.b = random_tensor(output_dim, init=init)

    def forward(self, input: Tensor) -> Tensor:
        # Save the input to use in the backward pass.
        self.input = input

        # Return the vector of neuron outputs.
        return [dot(input, self.w[o]) + self.b[o] for o in range(self.output_dim)]

    def backward(self, gradient: Tensor) -> Tensor:
        # Each b[o] gets added to output[o], which means
        # the gradient of b is the same as the output gradient.
        self.b_grad = gradient

        # Each w[o][i] multiplies input[i] and gets added to output[o].
        # So its gradient is input[i] * gradient[o].
        self.w_grad = [[self.input[i] * gradient[o] for i in range(self.input_dim)] for o in range(self.output_dim)]

        # Each input[i] multiplies every w[o][i] and gets added to every
        # output[o]. So its gradient is the sum of w[o][i] * gradient[o]
        # across all the outputs.
        return [sum(self.w[o][i] * gradient[o] for o in range(self.output_dim)) for i in range(self.input_dim)]

    def params(self) -> Iterable[Tensor]:
        return [self.w, self.b]

    def grads(self) -> Iterable[Tensor]:
        return [self.w_grad, self.b_grad]


In [20]:
class Sequential(Layer):
    """Una capa que consiste en una secuencia de otras capas. Depende de usted asegurarse de que la salida de cada capa Tiene sentido como la entrada a la siguiente capa."""

    def __init__(self, layers: List[Layer]) -> None:
        self.layers = layers
    
    def forward(self, input):
        """Simplemente reenvía la entrada a través de las capas en orden."""
        for layer in self.layers:
            input = layer.forward(input)
        return input
    
    def backward(self, gradient):
        """Simplemente retropropague el gradiente a través de las capas en reversa"""
        for layer in reversed(self.layers):
            gradient = layer.backward(gradient)
        return gradient
    
    def params(self) -> Iterable[Tensor]:
        """Solo devuelve los párametros de cada capa."""
        return (param for layer in self.layers for param in layer.params())
    
    def grads(self) -> Iterable[Tensor]:
        """Solo devuelva los grados de cada capa."""
        return (grad for layer in self.layers for grad in layer.grads())

In [21]:
xor_net = Sequential([Linear(input_dim= 2, output_dim= 2), Sigmoid(), Linear(input_dim=2, output_dim=1), Sigmoid()])

In [22]:
class Loss:

    def loss(self, predicted: Tensor, actual: Tensor) -> float:
        """¿Qué tan buenas son nuestras predicciones? (Los números más grandes son peores.)"""
        raise NotImplementedError
    
    def gradient(self, predicted: Tensor, actual: Tensor) -> Tensor:
        """¿Cómo cambia la pérdida a medida que cambian las predicciones?"""
        raise NotImplementedError

In [23]:
class SSE(Loss):
    """Función de pérdida que calcula la suma de los errores cuadrados."""

    def loss(self, predicted: Tensor, actual: Tensor) -> float:
        # calcul el tensor de diferencias al cuadrado
        squared_errors = tensor_combine(lambda predicted, actual: (predicted-actual) ** 2, predicted, actual)
        # y simplemente los suma
        return tensor_sum(squared_errors)
    
    def gradient(self, predicted: Tensor, actual: Tensor) -> Tensor:
        return tensor_combine(lambda predicted, actual: 2 * (predicted - actual), predicted, actual)

In [24]:
class Optimizer:
    """Un optimizador actualiza los pesos de una capa (en su lugar) utilizando información conocido por la capa o por el optimizador (o por ambos)."""
    def step(self, layer: Layer) -> None:
        raise NotImplementedError

In [25]:
class GradientDescent(Optimizer):

    def __init__(self, learning_rate: float = 0.1) -> None:
        self.lr = learning_rate

    def step(self, layer: Layer) -> None:
        for param, grad in zip(layer.params(), layer.grads()):
            # actualiza param usando un paso de gradiente
            param[:] = tensor_combine(lambda param, grad: param-grad * self.lr, param, grad)

In [26]:
tensor = [[1, 2],[3, 4]]
for row in tensor:
    row = [0, 0]

assert tensor == [[1, 2],[3, 4]], "assignment doesn't update la list"


for row in tensor:
    row[:] = [0, 0]

assert tensor == [[0, 0], [0, 0]], "but slice assignment does"
tensor

[[0, 0], [0, 0]]

In [27]:
class Momentum(Optimizer):

    def __init__(self, learning_rate: float, momentum: float = 0.9) -> None:
        self.lr = learning_rate
        self.mo = momentum
        self.updates: List[Tensor] = []

    def step(self, layer: Layer) -> None:
        # si no tenemos actualizaciones previas, empieza con todo ceros
        if not self.updates:
            self.updates = [zeros_like(grad) for grad in layer.grads()]
        for update, param, grad in zip(self.updates, layer.params(),layer.grads()):
            # aplica momentum
            update[:] = tensor_combine(lambda u, g: self.mo + (1 - self.mo) * g, update, grad)
            # luego da un paso de gradiente
            param[:] = tensor_combine(lambda p, u: p - self.lr * u, param, update)

XOR revisada

In [28]:
# datos de entrenamiento
xs = [[0., 0], [0., 1], [1., 0], [1., 1]]
ys = [[0.], [1.], [1.], [0.]]

In [29]:
random.seed(0)
net = Sequential([Linear(input_dim=2, output_dim=2), Sigmoid(), Linear(input_dim=2, output_dim=1)])

In [31]:
optimizer = GradientDescent(learning_rate=0.1)
loss = SSE()
with tqdm.trange(3000) as t:
    for epoch in t:
        epoch_loss = 0.0
        for x, y in zip(xs, ys):
            predicted = net.forward(x)
            epoch_loss += loss.loss(predicted, y)
            gradient = loss.gradient(predicted, y)
            net.backward(gradient)
            optimizer.step(net)
        t.set_description(f"xor loss {epoch_loss:.3f}")

xor loss 0.000: 100%|██████████| 3000/3000 [00:04<00:00, 612.48it/s]


In [32]:
for param in net.params():
    print(param)

[[-1.7240895194164054, -1.5229348698939176], [-5.489154401216733, -3.319209562855507]]
[1.688329195197331, 0.3891591933190046]
[[3.218959438021163, -3.620400982895372]]
[-0.5587660113096227]


In [40]:
def tanh(x: float) -> float:
    # Si x es muy grande o muy pequeño, tanh es (básicamente) 1 o -1.
    # Comprobamos esto porque, p.ej., math.exp(1000) da un error.
    if x < -100:  return -1
    elif x > 100: return 1

    em2x = math.exp(-2 * x)
    return (1 - em2x) / (1 + em2x)

class Tanh(Layer):
    class Tanh(Layer):
        def forward(self, input: Tensor) -> Tensor:
            # Aplicamos la función tangente hiperbólica: (e^x - e^-x) / (e^x + e^-x)
            self.tanh_values = tensor_apply(lambda x: math.tanh(x), input)
            return self.tanh_values

        def backward(self, gradient: Tensor) -> Tensor:
            # La derivada de tanh(x) es 1 - tanh(x)^2
            return tensor_combine(lambda x, grad: (1 - x ** 2) * grad, self.tanh_values, gradient)

class Relu(Layer):
    def forward(self, input: Tensor) -> Tensor:
        self.input = input
        return tensor_apply(lambda x: max(x, 0), input)

    def backward(self, gradient: Tensor) -> Tensor:
        return tensor_combine(lambda x, grad: grad if x > 0 else 0, self.input, gradient)

# Fizz Buzz revisado

In [36]:
def binary_encode(i: int, num_bits: int = 10) -> list:
    """Convierte un número i en una lista de num_bits (0 o 1)"""
    return [1 if i & (1 << n) else 0 for n in range(num_bits)]

def fizz_buzz_encode(n: int) -> list:
    """Genera la etiqueta (Target) para el entrenamiento"""
    if n % 15 == 0: return [0, 0, 0, 1] # fizzbuzz
    elif n % 5 == 0:  return [0, 0, 1, 0] # buzz
    elif n % 3 == 0:  return [0, 1, 0, 0] # fizz
    else:             return [1, 0, 0, 0] # número normal

def argmax(v: list) -> int:
    """Devuelve el índice del valor más alto en la lista"""
    return v.index(max(v))

In [38]:
xs = [binary_encode(n) for n in range(101, 1024)]
ys = [fizz_buzz_encode(n) for n in range(101, 1024)]

In [42]:
NUM_HIDDEN = 25
random.seed(0)

# Usamos las instancias de las clases Tanh y Sigmoid
net = Sequential([
    Linear(input_dim=10, output_dim=NUM_HIDDEN, init='uniform'),
    Tanh(),  
    Linear(input_dim=NUM_HIDDEN, output_dim=4, init='uniform'),
    Sigmoid()
])